# 02 · 구조화 로그와 요청 연결

목표: 로그를 JSON 필드로 조회하고 요청·추적 ID와 개인정보의 경계를 이해한다. 약12분.

**실행:** Python 3.11 공통 .venv 커널에서 위→아래. 다른 Notebook·서버·Docker·모델 설치 없이 독립 실행. 코드는 MOCK 고정 문장·대기만 사용한다. 실제 LLM 품질·처리량 측정이 아니다.

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
from week10.lab.experiments import lab_client

## 비교 요청

정상·검색 오류·잘못된 입력을 한 번씩 보낸다.

In [ ]:
async with lab_client() as (app, client):
    ok = await client.post("/api/v1/assist", json={"message": "SYNTHETIC-ONLY"})
    bad = await client.post(
        "/api/v1/assist", json={"message": "예시", "scenario": "retrieval_error"}
    )
    invalid = await client.post("/api/v1/assist", json={"message": " "})
    snapshot = app.state.telemetry.snapshot()
assert [ok.status_code, bad.status_code, invalid.status_code] == [200, 502, 422]

## 허용 필드

원문 대신 route·상태·시간·버전을 남긴다.

In [ ]:
logs = snapshot["logs"]
print(json.dumps(logs, ensure_ascii=False, indent=2))
assert "SYNTHETIC-ONLY" not in json.dumps(snapshot)

## 오류 조회

문자열 검색보다 필드 조건으로 필요한 이벤트를 좁힌다.

In [ ]:
errors = [r for r in logs if r["status"] >= 500]
assert len(errors) == 1
errors

## 사용자 문의와 연결

X-Request-ID로 해당 실행을 찾는다. 질문 자체를 로그에 복제할 필요가 없다.

In [ ]:
rid = bad.headers["X-Request-ID"]
matched = [r for r in logs if r["request_id"] == rid]
assert len(matched) == 1
matched[0]

## Trace 연결

같은 trace_id의 span을 모은다. ID는 인증·권한 증명이 아니다.

In [ ]:
tid = matched[0]["trace_id"]
spans = [s for s in snapshot["spans"] if s["trace_id"] == tid]
[(s["name"], s["status"]) for s in spans]

## 라벨 조합의 증가

예시 계산이다. 요청 ID를 지표 라벨에 넣을 때 1만 건만으로 조합 수가 급증한다.

In [ ]:
status_values, route_values, request_ids = 6, 4, 10_000
print("유한 상태·경로 상한:", status_values * route_values)
print("요청 ID까지 곱한 가상 상한:", status_values * route_values * request_ids)
assert "request_id" not in app.state.telemetry.prometheus()

## 메모리 보존 범위

200건 보관과 누적 완료 건수는 다른 값이다. 재시작하면 둘 다 초기화된다.

In [ ]:
print(snapshot["retention"])
assert set(logs[0]) == {
    "event",
    "timestamp",
    "request_id",
    "trace_id",
    "route",
    "status",
    "duration_ms",
    "release",
    "config_id",
}

## 프로젝트 적용 기록

예상 → 관찰 → 근거 → 다음 변경을 한 줄씩 기록한다. 실제로 실행하지 않은 조건은 미측정으로 표시한다. 학생 정보나 원문 입력은 공개 저장소에 올리지 않는다.